# D65-FairFace7-ROI Walkthrough
**Chart-free flash/no-flash cheek colorimetry with FairFace-routed sampling**

FitSkin / Pansor · August 2026

---

## How to run in Google Colab

1. **Runtime → Change runtime type → GPU** (optional; CPU works, FairFace is slower)
2. **Run Cell 1 (Setup)** — installs deps, clones repo, mounts Drive, downloads FairFace weights
3. Edit `PANSOR_ROOT` if your Drive folder name differs
4. Run remaining cells **top to bottom**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RooneyEmily/Fitskin/blob/main/d65_fairface7_roi_walkthrough.ipynb)

### What each stage does

| Stage | Input | Output | Why |
|---|---|---|---|
| Demosaic DNGs | flash + no-flash `.DNG` | linear RGB `A0`, `B0` | colorimetry needs linear sensor data, **no camera WB** |
| Apple cheek mask | `face_landmarks.json` in zip | boolean cheek ROI | same landmarks as Hybrid D65 Colab; **no MediaPipe** |
| Reflectance | `A0`, exposure-matched `B0` | `R0 = √(A0⊙B0')` | cancels ambient×flash to scene reflectance |
| Frozen Lab | `R0` + affine + 5500K→D65 | trimmed-mean Lab | claimable color path (~5.55 ΔE) |
| FairFace-7 | 8-bit crop from demosaiced `A0` | ethnicity prior | ROI routing only (not a colorimeter) |
| Specular-tone ROI | prior + cheek Lab cloud | final Lab | deployment sampling (~3.63 ΔE) |

> **Raw vs preview:** Lab always comes from **linear RAW demosaic**. The 8-bit preview is only so FairFace sees a normal photo-like face (it was trained that way).

> **Run Cell 1 first** after every Colab session restart.


## 0 — Setup (run this cell first every time)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# What this cell does:
#   1) install Python deps
#   2) clone / pull Fitskin from GitHub (code + calibration matrices)
#   3) mount Google Drive (Pansor DNGs + demographics xlsx)
#   4) download FairFace-7 weights once (~82 MB; not stored in git)
# ══════════════════════════════════════════════════════════════════════════════

# ── 1) Packages ───────────────────────────────────────────────────────────────
# rawpy     → demosaic DNG
# openpyxl  → demographics spreadsheet
# gdown     → FairFace weights from Google Drive
!pip install -q rawpy opencv-python-headless numpy matplotlib openpyxl gdown
try:
    import torch, torchvision  # noqa: F401  # usually already on Colab
except ImportError:
    !pip install -q torch torchvision

# ── 2) Clone / update Fitskin ─────────────────────────────────────────────────
# If the repo is private, git asks for a Personal Access Token (not your password).
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"  # change if using a fork
if os.path.isdir("Fitskin"):
    !cd Fitskin && git pull --ff-only || true
else:
    !git clone {REPO_URL}

REPO = Path("Fitskin").resolve()
assert (REPO / "models" / "fairface_race.py").is_file(), (
    f"Clone looks incomplete — missing models/fairface_race.py under {REPO}. "
    "Check REPO_URL / PAT, then re-run this cell."
)
assert (REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py").is_file(), (
    f"Clone looks incomplete — missing pansor eval script under {REPO}."
)

# ── 3) Mount Drive (Pansor data only) ─────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# Put Fitskin first on sys.path so `import models…` resolves here
sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

from delta_e_2000 import delta_e_2000
from flash_noflash_spectral import planck_xyz_y1
from models.fairface_race import FairFacePredictor, face_rgb_crop_from_landmarks
from scripts.evaluate_pansor20_chartfree_d65 import (
    D65,
    bradford_cat_matrix,
    linear_rgb_to_preview_bgr,
    load_dng_linear,
    load_apple_landmarks,
    apple_face_cheek_masks,
    match_flash_exposure,
    load_affine,
    load_demographics,
    discover_indoor_trials,
    extract_zip,
    mean_lab_on_mask,
)

# ── 4) Paths you may need to edit ─────────────────────────────────────────────
CAL_DIR = REPO / "calibration" / "tier3_affine"       # RGB→XYZ affine (frozen)
FAIRFACE_DIR = REPO / "calibration" / "fairface"      # .pt weights downloaded here
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)

# Indoor chart-free Pansor zips + demographics xlsx
# Expected layout on Drive:
#   <PANSOR_ROOT>/Pansor Dataset Demographics.xlsx
#   <PANSOR_ROOT>/Participant 1/*.zip
#   ...
# If your folder is elsewhere, set PANSOR_ROOT_OVERRIDE below.
PANSOR_ROOT_OVERRIDE = None  # e.g. Path("/content/drive/MyDrive/FitSkin/Pansor Dataset")

def _find_pansor_root() -> Path:
    if PANSOR_ROOT_OVERRIDE is not None:
        return Path(PANSOR_ROOT_OVERRIDE)
    drive_root = Path("/content/drive/MyDrive")
    candidates = [
        drive_root / "Pansor Dataset",
        drive_root / "FitSkin" / "Pansor Dataset",
        drive_root / "Fitskin" / "Pansor Dataset",
        drive_root / "Pansor",
        drive_root / "FitSkin" / "Pansor",
    ]
    # Also search a few levels for the demographics workbook
    hits = list(drive_root.glob("**/*Demographics*.xlsx"))[:20]
    for h in hits:
        # Prefer a folder that also has Participant * dirs
        parent = h.parent
        if any(parent.glob("Participant *")):
            return parent
    for c in candidates:
        if c.is_dir() and (
            (c / "Pansor Dataset Demographics.xlsx").is_file()
            or any(c.glob("Participant *"))
            or any(c.glob("**/*Demographics*.xlsx"))
        ):
            return c
    return candidates[0]  # default; Cell 2 will print diagnostics

PANSOR_ROOT = _find_pansor_root()

WORK_DIR = Path("/content/pansor_extract")              # unpacked zips (scratch)
OUT_DIR = Path("/content/d65_fairface7_roi_results")    # cohort CSV / TSV / summary
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Looking for Pansor data under Drive…")
print("  PANSOR_ROOT:", PANSOR_ROOT, "| exists:", PANSOR_ROOT.is_dir())
if PANSOR_ROOT.is_dir():
    print("  top entries:", sorted(p.name for p in PANSOR_ROOT.iterdir())[:12])
else:
    print("  Drive MyDrive sample:", sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20])

# ── 5) FairFace-7 weights (~82 MB, once per runtime) ──────────────────────────
FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"
assert FF7.is_file(), f"Missing FairFace weights: {FF7}"

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("PANSOR_ROOT exists:", PANSOR_ROOT.is_dir(), "→", PANSOR_ROOT)
print("Setup OK.")


## 1 — Load calibration, FairFace, demographics

**Frozen color path pieces (loaded once):**
- `tier3_affine` — pooled camera RGB→XYZ from chart training (not FitSkin cheeks)
- Planckian **5500 K** white → Bradford CAT → **D65** (fixed; no SCR illuminant estimate)
- FairFace-7 — race prior for ROI only
- Demographics xlsx — FitSkin Lab targets for ΔE scoring (not used at deployment)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Load the frozen color tools + FairFace + FitSkin targets
# ══════════════════════════════════════════════════════════════════════════════

# Camera RGB → XYZ affine (4×3 with bias column), from calibration/tier3_affine
M = load_affine(CAL_DIR)

# Fixed scene white for CAT: Planck locus at 5500 K, Y=1
xyz_w = planck_xyz_y1(5500.0, 0.0)
CAT = bradford_cat_matrix(xyz_w, D65)  # also applied inside mean_lab_on_mask
print("Affine M shape:", M.shape)
print("W_5500 XYZ:", np.round(xyz_w, 4))
print("D65 XYZ:", D65)

# FairFace-7 ResNet-34 (race head). Device = CUDA if available.
ff = FairFacePredictor.load(mode="7", weights_dir=FAIRFACE_DIR)
print("FairFace device:", ff.device)

# FitSkin Inside Lab targets (evaluation only — not fed into the color path)
DEMOG_XLSX = PANSOR_ROOT / "Pansor Dataset Demographics.xlsx"
if not DEMOG_XLSX.is_file():
    cands = list(PANSOR_ROOT.glob("**/*Demographics*.xlsx"))
    if not cands:
        cands = list(Path("/content/drive/MyDrive").glob("**/*Demographics*.xlsx"))
    DEMOG_XLSX = cands[0] if cands else DEMOG_XLSX
    if DEMOG_XLSX.is_file():
        # If we found the xlsx elsewhere, also move PANSOR_ROOT to its parent when it has zips
        parent = DEMOG_XLSX.parent
        if any(parent.glob("Participant *")):
            PANSOR_ROOT = parent
            print("Updated PANSOR_ROOT →", PANSOR_ROOT)

if not DEMOG_XLSX.is_file():
    drive = Path("/content/drive/MyDrive")
    print("ERROR: demographics xlsx not found.")
    print("  Searched under:", PANSOR_ROOT)
    print("  MyDrive folders:", sorted(p.name for p in drive.iterdir())[:30] if drive.is_dir() else "(Drive not mounted)")
    print("Fix: upload 'Pansor Dataset Demographics.xlsx' next to the Participant folders,")
    print("  then set in Cell 1:  PANSOR_ROOT_OVERRIDE = Path("/content/drive/MyDrive/<your folder>")")
    raise FileNotFoundError(
        "Need 'Pansor Dataset Demographics.xlsx' on Drive "
        "(same folder as 'Participant 1', 'Participant 2', …)."
    )

demo = load_demographics(DEMOG_XLSX)
print("Using demographics:", DEMOG_XLSX)
print(f"Demographics: {len(demo)} participants")
print("Example P1:", demo.get(1))


## 2 — Single-trial walkthrough

Pick one indoor zip (no bag / outside / light-box in the name).  
Cheek geometry comes from **Apple Vision** `face_landmarks.json` inside the zip.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Choose one indoor chart-free trial and unpack its zip
# Zip contents we need:
#   • no-flash DNG, flash DNG
#   • face_landmarks.json  (Apple Vision, full-res coordinates)
# ══════════════════════════════════════════════════════════════════════════════

trials = discover_indoor_trials(PANSOR_ROOT)
print(f"Indoor chart-free trials found: {len(trials)}")
assert len(trials) > 0, "No indoor zips found — check PANSOR_ROOT"

# Default = first trial. Uncomment an override to pick someone specific:
PARTICIPANT_ID = int(trials[0]["participant_id"])
TRIAL = int(trials[0]["trial"])
# PARTICIPANT_ID, TRIAL = 6, 1   # Shuyi
# PARTICIPANT_ID, TRIAL = 1, 1   # Bryan

t = next(
    x for x in trials
    if int(x["participant_id"]) == PARTICIPANT_ID and int(x["trial"]) == TRIAL
)
meta = demo[PARTICIPANT_ID]
print("Selected:", t["subject_id"], meta["name"], meta["ethnicity"])
print("Zip:", t["zip_path"])
print("FitSkin Lab (target):", meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"])

ZIP_PATH = Path(t["zip_path"])
work = WORK_DIR / t["subject_id"]
nf, fl, lm_path = extract_zip(ZIP_PATH, work)
print("Extracted files:", nf.name, "|", fl.name, "|", lm_path.name)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — RAW demosaic → cheek mask → flash/no-flash reflectance
#
# Colorimetry path (linear RAW):
#   A0, B0  = demosaic WITHOUT camera white-balance
#   cheek   = Apple landmark polygons scaled to demosaic resolution
#   B0m     = flash frame exposure-matched to no-flash on the cheek
#   R0      = √(A0 ⊙ B0m)   ≈ reflectance (ambient×flash cancel)
#
# Preview path (display only / FairFace input):
#   8-bit stretch of A0 — NOT used for Lab
# ══════════════════════════════════════════════════════════════════════════════

# half_size=True → faster / less RAM on Colab; still linear RAW
A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
B0 = load_dng_linear(fl, half_size=True, use_camera_wb=False)
if B0.shape != A0.shape:
    B0 = cv2.resize(B0, (A0.shape[1], A0.shape[0]), interpolation=cv2.INTER_AREA)

# Landmarks are stored at full capture size; helper rescales polygons to A0 shape
lm = load_apple_landmarks(lm_path)
_, cheek = apple_face_cheek_masks(lm, A0.shape[0], A0.shape[1])
print("Demosaic shape:", A0.shape, "| cheek pixels:", int(np.count_nonzero(cheek)))

# Match mean cheek luma of flash to no-flash, then geometric-mean reflectance
B0m, s = match_flash_exposure(A0, B0, cheek)
R0 = np.sqrt(np.maximum(A0, 0) * np.maximum(B0m, 0) + 1e-8)
print(f"Flash exposure scale s={s:.4f}  (B0' = s · B0)")

# 8-bit preview from linear A0 (FairFace + plots only)
preview = linear_rgb_to_preview_bgr(A0)
overlay = preview.copy()
overlay[cheek > 0] = (0.6 * overlay[cheek > 0] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
ax[0].set_title("No-flash preview (from RAW)"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
ax[1].set_title("Apple cheek mask"); ax[1].axis("off")
ax[2].imshow(np.clip(R0 ** (1 / 2.2), 0, 1))
ax[2].set_title("Reflectance R0 (γ for display)"); ax[2].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Frozen color path (claimable): trimmed-mean cheek Lab
#   R0 → affine RGB→XYZ → Bradford CAT (5500K→D65) → CIELAB
#   l_sampling="off" → 5% trimmed mean on cheek (no ethnicity / FairFace)
# Compare to FitSkin Inside Lab with CIEDE2000.
# ══════════════════════════════════════════════════════════════════════════════

fit = np.array([meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"]], dtype=np.float64)

Lab_frozen, _ = mean_lab_on_mask(
    R0, cheek, M,
    xyz_scene_white=xyz_w,   # triggers 5500K→D65 CAT inside
    cat_degree=1.0,
    l_sampling="off",        # trimmed mean only
)
de_frozen = float(delta_e_2000(Lab_frozen, fit))
print(f"Frozen Lab = ({Lab_frozen[0]:.1f}, {Lab_frozen[1]:.1f}, {Lab_frozen[2]:.1f})")
print(f"FitSkin    = ({fit[0]:.1f}, {fit[1]:.1f}, {fit[2]:.1f})")
print(f"ΔE00 frozen (no ROI heuristic) = {de_frozen:.2f}")
print("(Cohort mean for this path is ~5.55)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Deployment ROI: FairFace-7 prior → specular_tone sampling
#
# FairFace sees an 8-bit face crop (from RAW demosaic preview + Apple landmarks).
# It does NOT change XYZ/Lab math — only which cheek pixels / L* rule we use.
#
# Mapping (FairFace → Pansor ROI key):
#   Black/White/Indian as-is
#   East/SE Asian → Asian
#   Middle Eastern → Iranian
#   Latino_Hispanic → Asian
# ══════════════════════════════════════════════════════════════════════════════

face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)
ff_out = ff.predict_rgb(face_rgb)

print("FairFace-7 raw label:", ff_out["fairface_label"], f"(conf={ff_out['confidence']:.2f})")
print("→ ROI ethnicity key: ", ff_out["predicted_ethnicity"])
print("Class probs:", {k: round(v, 3) for k, v in ff_out["race_probs"].items()})

plt.figure(figsize=(3, 3))
plt.imshow(face_rgb)
plt.title(f"FF crop → {ff_out['fairface_label']}")
plt.axis("off")
plt.show()

# Same color math as Cell 5, but cheek aggregation follows specular_tone rules
# conditioned on the FairFace prior (deployment — no demographics labels).
Lab_ff, sm = mean_lab_on_mask(
    R0, cheek, M,
    xyz_scene_white=xyz_w,
    cat_degree=1.0,
    l_sampling="specular_tone",
    ethnicity=ff_out["predicted_ethnicity"],
)
de_ff = float(delta_e_2000(Lab_ff, fit))

# Oracle: same ROI rules but with true demographics ethnicity (not available at deploy)
Lab_oracle, _ = mean_lab_on_mask(
    R0, cheek, M,
    xyz_scene_white=xyz_w,
    cat_degree=1.0,
    l_sampling="specular_tone",
    ethnicity=meta["ethnicity"],
)
de_oracle = float(delta_e_2000(Lab_oracle, fit))

print(f"\nD65-FairFace7-ROI = ({Lab_ff[0]:.1f}, {Lab_ff[1]:.1f}, {Lab_ff[2]:.1f})  ΔE00={de_ff:.2f}")
print(f"Oracle demographics= ({Lab_oracle[0]:.1f}, {Lab_oracle[1]:.1f}, {Lab_oracle[2]:.1f})  ΔE00={de_oracle:.2f}")
print(f"Frozen trimmed mean= ({Lab_frozen[0]:.1f}, {Lab_frozen[1]:.1f}, {Lab_frozen[2]:.1f})  ΔE00={de_frozen:.2f}")
print("Which specular_tone branch fired:", sm)


## 3 — Full indoor chart-free cohort (optional)

Runs the production CLI on all indoor zips (`N ≈ 65`). Writes Emily-format TSV + `summary.json`.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Full cohort: same stack as Cells 4–6 for every indoor trial
#   --scr-mode preawb_cat --fixed-cat-k 5500   → frozen color path
#   --l-sampling fairface7                     → FairFace ROI prior
#   --emily-tsv                                → Participant_ID / Trial / Lab / ΔE table
# ══════════════════════════════════════════════════════════════════════════════

cmd = [
    "python3", str(REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py"),
    "--data-root", str(PANSOR_ROOT),
    "--demographics", str(DEMOG_XLSX),
    "--cal-dir", str(CAL_DIR),
    "--scr-mode", "preawb_cat",
    "--fixed-cat-k", "5500",
    "--l-sampling", "fairface7",
    "--fairface-dir", str(FAIRFACE_DIR),
    "--emily-tsv",
    "--work-dir", str(WORK_DIR),
    "--out-dir", str(OUT_DIR),
]
print(" ".join(cmd))
subprocess.check_call(cmd)

summary = json.loads((OUT_DIR / "summary.json").read_text())
print("\n=== D65-FairFace7-ROI ===")
print(f"n={summary['n_trials']}  mean={summary['mean_de00']:.2f}  median={summary['median_de00']:.2f}")
print(f"{'Ethnicity':10s} {'n':>4s} {'mean':>8s} {'median':>8s}")
for eth, st in summary["by_ethnicity"].items():
    print(f"{eth:10s} {st['n']:4d} {st['mean_de00']:8.2f} {st['median_de00']:8.2f}")
print("Emily TSV:", OUT_DIR / "table_emily_format.tsv")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Preview Emily table and download artifacts to your laptop
# ══════════════════════════════════════════════════════════════════════════════

tsv = OUT_DIR / "table_emily_format.tsv"
print(tsv.read_text())

from google.colab import files
files.download(str(tsv))
files.download(str(OUT_DIR / "summary.json"))


## 4 — Reference numbers (pinned)

| Method | Runtime labels? | Mean ΔE₀₀ |
|---|---|---:|
| Camera-WB + affine gate | — | ~5.93 |
| Frozen `preawb_cat` (sampling off) | — | **5.55** |
| Lab tone→ethnicity ROI (LOSO) | no | 3.89 |
| **D65-FairFace7-ROI** | **no** | **3.63** |
| Demographics + ROI (oracle) | yes | 3.23 |

### CLI (local)
```bash
python3 scripts/evaluate_pansor20_chartfree_d65.py \
  --scr-mode preawb_cat --fixed-cat-k 5500 \
  --l-sampling fairface7 --emily-tsv \
  --out-dir results/pansor20_fairface7
```

Methods write-up: `docs/d65_fairface7_roi_overleaf.tex`
